# 03b_coref — Coreference resolution (fastcoref)

> **Environment:** requires the project venv **`.venv311`** (Python 3.11) as the Jupyter kernel — the pipeline dependencies are installed only there. `run_all.command` uses it automatically. See `README.md` → Environment setup.

**Input:** `data/interim/sentences.jsonl` (+ `data/interim/corpus_clean.jsonl` for article bodies)
**Output:** `data/interim/sentences_coref.jsonl` (same schema as `sentences.jsonl` + one field `coref_resolved_text`)

Pronouns ("he", "she", "it", "they") carry actors that NER misses. This notebook resolves them with **fastcoref** at the **article level** (pronouns refer across sentences), then re-aligns the resolved spans back onto each sentence by character offset.

A pronoun is rewritten **only when its coreference cluster has an antecedent that resolves to a whitelisted actor** (`ACTOR_WHITELIST` in `src/alias_map.py`). `it`/`they` whose antecedent is not a whitelisted actor are left untouched — deliberate **partial coverage**. The original `text` is preserved verbatim; the rewritten version goes to `coref_resolved_text` (equal to `text` when nothing was resolved). Notebook 04 runs NER on `coref_resolved_text`.

## Pipeline steps in this notebook

1. Setup & paths
2. Load the fastcoref model
3. Load sentences + article bodies; import actor whitelist / alias map
4. Coreference helpers (pronoun set, actor resolver, per-article processor)
5. Resolve every article (per-article try/except — never crash the run)
6. Re-align resolved spans onto sentence boundaries (character offsets)
7. Quality report (pronouns found / resolved / skipped; sample rewrites)
8. Write sentences_coref.jsonl

## Step 1: Setup & paths

In [ ]:
import json
import sys
import time
import warnings
import logging
import contextlib
import io
from pathlib import Path
from collections import Counter, defaultdict

# fastcoref / torch are chatty; keep the executed notebook readable.
warnings.filterwarnings('ignore')
logging.disable(logging.WARNING)

# Silence tqdm bars: fastcoref emits one per predict call, which under
# nbconvert renders as thousands of display frames and bloats the notebook.
from functools import partialmethod
import tqdm as _tqdm
_tqdm.tqdm.__init__ = partialmethod(_tqdm.tqdm.__init__, disable=True)
try:
    import tqdm.auto as _tqdm_auto
    _tqdm_auto.tqdm.__init__ = partialmethod(_tqdm_auto.tqdm.__init__, disable=True)
except Exception:
    pass

_cwd = Path().resolve()
ROOT = next(
    (p for p in [_cwd] + list(_cwd.parents) if (p / 'src').is_dir()),
    _cwd,
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

INTERIM_DIR = ROOT / 'data' / 'interim'
SENT_FILE   = INTERIM_DIR / 'sentences.jsonl'
CLEAN_FILE  = INTERIM_DIR / 'corpus_clean.jsonl'
OUT_FILE    = INTERIM_DIR / 'sentences_coref.jsonl'

print(f'Python    : {sys.executable}')
print(f'Sentences : {SENT_FILE}')
print(f'Bodies    : {CLEAN_FILE}')
print(f'Output    : {OUT_FILE}')
assert (ROOT / 'src').is_dir(), f'ERROR: src/ not found under {ROOT}'
assert SENT_FILE.exists(),  f'ERROR: {SENT_FILE} not found — run 03_preprocess first'
assert CLEAN_FILE.exists(), f'ERROR: {CLEAN_FILE} not found — run 02_clean first' 

## Step 2: Load the fastcoref model

`FCoref` is the fast (distilled) coreference model. It loads from the local
Hugging Face cache — no download needed once `pip install fastcoref` has run.
CPU is fine; this corpus resolves in a couple of minutes.

In [ ]:
from fastcoref import FCoref

model = FCoref(device='cpu')
print('Loaded FCoref model')

## Step 3: Load sentences, article bodies, and the actor whitelist

We need the **article bodies** (from `corpus_clean.jsonl`) because coreference is
resolved per article, then mapped back to sentences by character offset. The
sentence records themselves come from `03_preprocess`.

In [ ]:
from src.alias_map import ALIAS_MAP, ACTOR_WHITELIST

with open(SENT_FILE, encoding='utf-8') as f:
    sentences = [json.loads(line) for line in f]

with open(CLEAN_FILE, encoding='utf-8') as f:
    bodies = {a['id']: a['body'] for a in (json.loads(line) for line in f)}

# Group sentence records by article, preserving their order in the file.
sents_by_article = defaultdict(list)
for s in sentences:
    sents_by_article[s['article_id']].append(s)

print(f'Loaded {len(sentences)} sentences across {len(sents_by_article)} articles')
print(f'Article bodies available : {len(bodies)}')
print(f'ACTOR_WHITELIST entries  : {len(ACTOR_WHITELIST)}')

## Step 4: Coreference helpers

- `PRONOUNS` — the third-person pronouns we are willing to rewrite. First/second
  person ("I", "we", "you") are excluded: in quoted speech they refer to the
  speaker, not to a narrative actor.
- `resolve_actor()` mirrors notebook 04's `extract_actors` rule exactly, so an
  antecedent counts as a "whitelisted actor" by the same logic the network uses.
- `article_replacements()` runs coref on one body and returns, for each pronoun
  whose cluster antecedent is a whitelisted actor, a `(start, end, surface)`
  replacement in body-character coordinates.

In [ ]:
PRONOUNS = {
    'he', 'him', 'his', 'himself',
    'she', 'her', 'hers', 'herself',
    'it', 'its', 'itself',
    'they', 'them', 'their', 'theirs', 'themselves',
}

def resolve_actor(surface):
    """Surface form -> canonical actor ID, or None if not a whitelisted actor.
    Same rule as notebook 04's extract_actors (alias lookup, else upper-snake,
    drop EVENT_ANCHOR, keep only ACTOR_WHITELIST)."""
    canonical = ALIAS_MAP.get(surface.lower().strip())
    if canonical is None:
        canonical = surface.upper().replace(' ', '_')
    if canonical == 'EVENT_ANCHOR':
        return None
    return canonical if canonical in ACTOR_WHITELIST else None

def is_pronoun(text):
    return text.lower().strip() in PRONOUNS

def article_replacements(body, preds):
    """From one article's coref prediction, build pronoun replacements.

    Returns (replacements, found, resolved, skipped):
      replacements : list of (start, end, antecedent_surface) in body coords
      found        : pronoun mentions seen inside clusters
      resolved     : pronoun mentions we will rewrite (whitelisted antecedent)
      skipped      : pronoun mentions left as-is (no whitelisted antecedent)
    """
    replacements = []
    found = resolved = skipped = 0
    for cluster in preds.get_clusters(as_strings=False):
        mentions = [(s, e, body[s:e]) for s, e in cluster]
        pron = [m for m in mentions if is_pronoun(m[2])]
        if not pron:
            continue
        found += len(pron)
        # Antecedent = first non-pronoun mention (document order) that resolves
        # to a whitelisted actor.
        antecedent = None
        for s, e, txt in mentions:
            if not is_pronoun(txt) and resolve_actor(txt):
                antecedent = txt
                break
        if antecedent is None:
            skipped += len(pron)
            continue
        for s, e, _txt in pron:
            replacements.append((s, e, antecedent))
        resolved += len(pron)
    return replacements, found, resolved, skipped

## Step 5: Resolve every article

Coref is run per article inside a `try/except`: if fastcoref raises on one
article we log its `article_id` and continue — a single bad document never
crashes the whole run. (fastcoref's own progress bars are sent to stderr, which
we swallow here to keep the output clean; our progress prints go to stdout.)

In [ ]:
article_ids = list(sents_by_article)
article_repls = {aid: [] for aid in article_ids}
tot_found = tot_resolved = tot_skipped = 0
errored = []

t0 = time.time()
with contextlib.redirect_stderr(io.StringIO()):
    for i, aid in enumerate(article_ids):
        body = bodies.get(aid)
        if not body:
            continue
        try:
            preds = model.predict(texts=[body])[0]
            repls, found, res, skip = article_replacements(body, preds)
        except Exception as exc:           # noqa: BLE001 — never crash the run
            errored.append((aid, type(exc).__name__))
            continue
        article_repls[aid] = repls
        tot_found += found
        tot_resolved += res
        tot_skipped += skip
        if (i + 1) % 50 == 0 or i == 0:
            print(f'  article {i + 1:4d}/{len(article_ids)}  '
                  f'elapsed={time.time() - t0:.0f}s')

print(f'\nCoref done in {time.time() - t0:.0f}s')
print(f'Pronoun mentions found       : {tot_found}')
print(f'  resolved (whitelisted)     : {tot_resolved}')
print(f'  skipped (no wl antecedent) : {tot_skipped}')
print(f'Articles errored             : {len(errored)}  {errored[:10]}')

## Step 6: Re-align resolved spans onto sentences

fastcoref gives replacement spans in **article-body** coordinates. Each stored
sentence is a (stripped) substring of its body, so we locate it with a running
`find` cursor to get its character span, then apply only the replacements that
fall inside that span — offset-adjusted and applied right-to-left so earlier
offsets stay valid. Every sentence gets a `coref_resolved_text` (equal to `text`
when nothing changed).

In [ ]:
def apply_to_sentence(sent_text, sent_start, repls):
    """Apply body-coordinate replacements that fall within this sentence."""
    end = sent_start + len(sent_text)
    local = [(s - sent_start, e - sent_start, surf)
             for (s, e, surf) in repls
             if s >= sent_start and e <= end]
    if not local:
        return sent_text
    out = sent_text
    for s, e, surf in sorted(local, key=lambda x: x[0], reverse=True):
        out = out[:s] + surf + out[e:]
    return out

n_changed = 0
n_unaligned = 0
for aid, sents in sents_by_article.items():
    body = bodies.get(aid, '')
    repls = article_repls.get(aid, [])
    cursor = 0
    for s in sents:
        txt = s['text']
        start = body.find(txt, cursor) if body else -1
        if start == -1:
            s['coref_resolved_text'] = txt      # cannot align -> leave unchanged
            n_unaligned += 1
            continue
        cursor = start + len(txt)
        resolved_txt = apply_to_sentence(txt, start, repls) if repls else txt
        s['coref_resolved_text'] = resolved_txt
        if resolved_txt != txt:
            n_changed += 1

print(f'Sentences rewritten by coref : {n_changed} / {len(sentences)} '
      f'({100 * n_changed / max(len(sentences), 1):.1f}%)')
print(f'Sentences not alignable      : {n_unaligned}  (kept = original text)')

## Step 7: Quality report — sample rewrites

In [ ]:
import random
random.seed(42)

changed = [s for s in sentences if s['coref_resolved_text'] != s['text']]
print(f'Total rewritten sentences: {len(changed)}')
print('\n--- Sample coref rewrites (before -> after) ---')
for s in random.sample(changed, min(8, len(changed))):
    print(f"\n  [{s['sentence_id']}]  window={s['window']}")
    print(f"   before : {s['text'][:200]}")
    print(f"   after  : {s['coref_resolved_text'][:200]}")

## Step 8: Write sentences_coref.jsonl

In [ ]:
with open(OUT_FILE, 'w', encoding='utf-8') as f:
    for s in sentences:
        f.write(json.dumps(s, ensure_ascii=False) + '\n')

print(f'Wrote {len(sentences)} records to {OUT_FILE}')
print()
print('VALIDATION CHECKPOINT (03b_coref):')
print(f'  Sentences              : {len(sentences)}')
print(f'  Pronouns found         : {tot_found}')
print(f'  Pronouns resolved      : {tot_resolved}')
print(f'  Pronouns skipped       : {tot_skipped}')
print(f'  Sentences rewritten    : {n_changed}')
print(f'  Articles errored       : {len(errored)}')
print()
print('NOTE: it/they whose antecedent is not a whitelisted actor are left')
print('      unresolved by design (partial coverage). Notebook 04 runs NER on')
print('      coref_resolved_text and falls back to text when unchanged.')